# تمرین ۱ — Fine-tuning برای تحلیل احساسات فارسی

نوت‌بوک Colab. آپلود کن در colab.research.google.com و سلول‌ها رو یکی‌یکی با Shift+Enter اجرا کن.

**هدف:** دو مدل pretrained (ParsBERT و XLM-RoBERTa) رو روی نظرات اسنپ‌فود fine-tune می‌کنیم برای تشخیص مثبت/منفی، و دقتشون رو مقایسه می‌کنیم.

**چرا:** ParsBERT تک‌زبانه‌ی فارسیه، XLM-RoBERTa چندزبانه — می‌خوایم ببینیم این تفاوت چه اثری روی دقت داره.


## مرحله‌ی ۰ — فعال‌سازی GPU

`Runtime → Change runtime type → T4 GPU → Save`. بعد سلول زیر رو اجرا کن؛ اگه اسم یک GPU (مثلاً Tesla T4) نشون داد، یعنی فعاله.

In [ ]:
# بررسی اینکه آیا GPU در دسترس است یا نه
!nvidia-smi

## مرحله‌ی ۱ — نصب کتابخونه‌ها

| کتابخونه | نقش |
|---|---|
| `transformers` | مدل‌های pretrained + کلاس `Trainer` |
| `datasets` | دانلود دیتاست از HuggingFace |
| `accelerate` | اجرای بهینه‌ی `Trainer` روی GPU |
| `evaluate` | محاسبه‌ی Accuracy و F1 |
| `scikit-learn` | ابزارهای کمکی ML |

علامت `!` یعنی دستور شل، نه پایتون.

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

## مرحله‌ی ۲ — Import کتابخونه‌ها

In [ ]:
import numpy as np
import pandas as pd
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

print("همه‌ی کتابخونه‌ها با موفقیت import شدند.")

## مرحله‌ی ۳ — بارگذاری دیتاست

دیتاست پیشنهادی: نظرات اسنپ‌فود (مثبت/منفی).

⚠️ **این قسمت رو باید خودت چک کنی:** شناسه‌ی دقیق دیتاست ممکنه فرق داشته باشه. برو به huggingface.co/datasets و کلمه‌ی `snappfood` رو سرچ کن، شناسه‌ی دقیق رو کپی کن و در `DATASET_NAME` زیر جایگزین کن.

In [ ]:
DATASET_NAME = "PNLPhub/snappfood-sentiment-analysis"  # اگر شناسه فرق داشت، اینجا عوضش کن

dataset = load_dataset(DATASET_NAME)
print(dataset)

خروجی `dataset` یک `DatasetDict` با split‌ها (مثلاً train/test) و ستون‌هاست. حالا اسم دقیق ستون‌ها رو ببینیم، چون بین دیتاست‌ها فرق می‌کنه.

In [ ]:
# اسم دقیق ستون‌ها را ببینیم
split_name = list(dataset.keys())[0]   # اولین split موجود (معمولاً "train")
print("ستون‌های موجود:", dataset[split_name].column_names)
print("\nیک نمونه از داده:")
print(dataset[split_name][0])

⚠️ **این قسمت رو خودت پر کن:** بر اساس خروجی بالا، اسم ستون متن و اسم ستون برچسب رو در دو متغیر زیر بذار. برای این دیتاست، ستون `label` رشته‌ایه (`'SAD'`, `'HAPPY'`) و مدل نمی‌تونه روش train بشه — باید از `label_id` (که عدده) استفاده کنیم و بعد تبدیلش کنیم به int و اسمش رو بذاریم `labels` (اسمی که `Trainer` انتظار داره).

In [ ]:
TEXT_COLUMN = "comment"     # اگر اسم ستون متن چیز دیگری بود، اینجا عوضش کن
LABEL_COLUMN = "label_id"   # مهم: از "label_id" استفاده کن نه "label"، چون "label" رشته‌ست (مثل 'SAD') و مدل فقط عدد می‌فهمه

from datasets import Value

# label_id به‌صورت float ذخیره شده (1.0, 0.0)؛ باید به int تبدیل بشه وگرنه مدل اون رو رگرسیون/multi-label حساب می‌کنه
dataset = dataset.cast_column(LABEL_COLUMN, Value("int64"))

# ستون رشته‌ای "label" رو حذف می‌کنیم و label_id رو به اسمی که Trainer انتظار داره ("labels") تغییر نام می‌دیم
# این کار رو همین‌جا، قبل از تقسیم train/test و tokenization انجام می‌دیم تا لازم نباشه بعداً برای هر دو مدل تکرارش کنیم
if "label" in dataset[split_name].column_names:
    dataset = dataset.remove_columns(["label"])
dataset = dataset.rename_column(LABEL_COLUMN, "labels")
LABEL_COLUMN = "labels"

# تعداد کلاس‌ها (مثلاً 2 برای مثبت/منفی) را از روی داده حساب می‌کنیم
num_labels = len(set(dataset[split_name][LABEL_COLUMN]))
print("تعداد کلاس‌ها:", num_labels)

## مرحله‌ی ۴ — تقسیم train / test

اگه دیتاست از قبل split جدا نداشته باشه، ۲۰٪ رو برای test کنار می‌ذاریم. `seed=42` برای تکرارپذیری و مقایسه‌ی منصفانه‌ی دو مدله.

In [ ]:
if "test" in dataset:
    train_data = dataset["train"]
    test_data = dataset["test"]
else:
    split_dataset = dataset[split_name].train_test_split(test_size=0.2, seed=42)
    train_data = split_dataset["train"]
    test_data = split_dataset["test"]

print("تعداد نمونه‌های آموزش:", len(train_data))
print("تعداد نمونه‌های آزمون:", len(test_data))

## مرحله‌ی ۵ — Tokenization

متن رو به دنباله‌ای از عدد (token id) تبدیل می‌کنیم. `padding` یعنی جمله‌های کوتاه‌تر پر می‌شن تا هم‌طول بشن؛ `truncation` یعنی جمله‌های بلندتر از `max_length` بریده می‌شن.

In [ ]:
MODEL_NAME_1 = "HooshvareLab/bert-fa-base-uncased"   # ParsBERT — مدل اول

tokenizer_1 = AutoTokenizer.from_pretrained(MODEL_NAME_1)

def make_tokenize_fn(tokenizer):
    def tokenize_fn(batch):
        return tokenizer(
            batch[TEXT_COLUMN],
            padding="max_length",
            truncation=True,
            max_length=128,
        )
    return tokenize_fn

tokenize_fn_1 = make_tokenize_fn(tokenizer_1)

train_tokenized_1 = train_data.map(tokenize_fn_1, batched=True)
test_tokenized_1 = test_data.map(tokenize_fn_1, batched=True)

print(train_tokenized_1[0].keys())

خروجی شامل `input_ids` و `attention_mask` است — همون فرمتی که مدل نیاز داره.

## مرحله‌ی ۶ — بارگذاری مدل

`AutoModelForSequenceClassification` مدل pretrained رو می‌گیره و یک لایه‌ی طبقه‌بندی جدید (با خروجی به تعداد `num_labels`) روی آخرش اضافه می‌کنه. این لایه‌ی جدید همونیه که در fine-tuning یاد می‌گیره.

In [ ]:
model_1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_1,
    num_labels=num_labels,
    problem_type="single_label_classification",  # صراحتاً می‌گیم طبقه‌بندی تک‌برچسبیه (نه رگرسیون/multi-label)
)

## مرحله‌ی ۷ — معیار ارزیابی

**Accuracy:** درصد پیش‌بینی‌های درست. **F1-score:** ترکیب Precision و Recall — برای داده‌ی نامتوازن معیار بهتریه.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

## مرحله‌ی ۸ — `TrainingArguments` و `Trainer`

| پارامتر | معنی |
|---|---|
| `learning_rate` | نرخ یادگیری (کوچیک، تا دانش قبلی مدل خراب نشه) |
| `per_device_train_batch_size` | اندازه‌ی batch |
| `num_train_epochs` | چندبار مدل کل داده رو می‌بینه |

`Trainer` کل حلقه‌ی آموزش (forward → loss → backward → آپدیت وزن‌ها) رو خودش مدیریت می‌کنه.

In [ ]:
training_args_1 = TrainingArguments(
    output_dir="./results_model1",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none",   # جلوگیری از اتصال خودکار به سرویس‌های لاگ بیرونی
)

trainer_1 = Trainer(
    model=model_1,
    args=training_args_1,
    train_dataset=train_tokenized_1,
    eval_dataset=test_tokenized_1,
    compute_metrics=compute_metrics,
)

## مرحله‌ی ۹ — آموزش مدل اول

⏱ چند دقیقه تا حدود یک ساعت طول می‌کشه (بسته به GPU). برای تست سریع: `train_data.select(range(5000))`، ولی برای گزارش نهایی روی کل داده اجرا کن.


In [ ]:
trainer_1.train()

## مرحله‌ی ۱۰ — ارزیابی مدل اول

نتیجه رو نگه می‌داریم تا بعداً با مدل دوم مقایسه کنیم.


In [ ]:
import time

start_time = time.time()
results_model1 = trainer_1.evaluate()
elapsed_model1 = time.time() - start_time

print(results_model1)
print(f"\nزمان تقریبی این اجرا (ارزیابی): {elapsed_model1:.1f} ثانیه")
print("زمان کامل آموزش را از لاگ‌های بالای سلول trainer_1.train() (ستون Runtime) هم می‌توانی ببینی.")

## مرحله‌ی ۱۱ — مدل دوم (XLM-RoBERTa)

همه‌ی تنظیمات عین مدل اول می‌مونن؛ فقط خود مدل عوض می‌شه.


In [ ]:
MODEL_NAME_2 = "xlm-roberta-base"   # XLM-RoBERTa — مدل دوم

# ۵) توکنایز کردن داده با tokenizer مدل دوم
tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_NAME_2)
tokenize_fn_2 = make_tokenize_fn(tokenizer_2)

train_tokenized_2 = train_data.map(tokenize_fn_2, batched=True)
test_tokenized_2 = test_data.map(tokenize_fn_2, batched=True)

# ۶) بارگذاری مدل دوم برای طبقه‌بندی
model_2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_2,
    num_labels=num_labels,
    problem_type="single_label_classification",
)

# ۸) همان تنظیمات آموزشی، فقط output_dir را جدا می‌کنیم که با مدل اول قاطی نشود
training_args_2 = TrainingArguments(
    output_dir="./results_model2",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none",
)

trainer_2 = Trainer(
    model=model_2,
    args=training_args_2,
    train_dataset=train_tokenized_2,
    eval_dataset=test_tokenized_2,
    compute_metrics=compute_metrics,
)

# ۹) آموزش مدل دوم
trainer_2.train()

In [ ]:
# ۱۰) ارزیابی نهایی مدل دوم
start_time = time.time()
results_model2 = trainer_2.evaluate()
elapsed_model2 = time.time() - start_time

print(results_model2)
print(f"\nزمان تقریبی این اجرا (ارزیابی): {elapsed_model2:.1f} ثانیه")

## مرحله‌ی ۱۲ — جدول مقایسه

این جدول رو مستقیم در گزارش کپی کن.


In [ ]:
comparison_table = pd.DataFrame([
    {
        "دیتاست": DATASET_NAME,
        "مدل": "ParsBERT",
        "Accuracy": round(results_model1["eval_accuracy"], 4),
        "F1-score": round(results_model1["eval_f1"], 4),
    },
    {
        "دیتاست": DATASET_NAME,
        "مدل": "XLM-RoBERTa",
        "Accuracy": round(results_model2["eval_accuracy"], 4),
        "F1-score": round(results_model2["eval_f1"], 4),
    },
])

comparison_table

**نتایج واقعی این اجرا** (برای مرجع، در صورتی که نوت‌بوک دوباره اجرا نشود):

| دیتاست | مدل | Accuracy | F1-score |
|---|---|---|---|
| PNLPhub/snappfood-sentiment-analysis | ParsBERT | 0.8680 | 0.8679 |
| PNLPhub/snappfood-sentiment-analysis | XLM-RoBERTa | 0.8776 | 0.8773 |


## مرحله‌ی ۱۳ (اختیاری) — دیتاست دوم

مراحل ۳ تا ۱۲ رو با یک `DATASET_NAME` دیگه دوباره اجرا کن.


## مرحله‌ی ۱۴ — پاسخ سوالات تحلیلی

**۱. کدوم مدل دقت بالاتری داشت؟ چرا؟**

با نتایج واقعی این اجرا، XLM-RoBERTa (Accuracy 0.8776 / F1 0.8773) کمی از ParsBERT (Accuracy 0.8680 / F1 0.8679) بهتر بود — برخلاف انتظار رایج که مدل تک‌زبانه‌ی فارسی برتری داشته باشه. دلیل اصلی، overfitting سریع‌تر ParsBERT است: بین epoch ۲ و ۳ validation loss آن از 0.323 به 0.426 جهش کرد و accuracy از 0.8737 به 0.8680 افت کرد. اگر checkpoint بهینه (epoch ۲) انتخاب می‌شد فاصله‌ی دو مدل خیلی کمتر بود. دو مدل هم در یک سشن واحد و پشت‌سرهم اجرا نشدند، پس تفاوت جزئی محیط اجرا هم می‌تواند اثر داشته باشد.

**۲. نتیجه روی دو دیتاست یکسان بود یا فرق داشت؟**

این مقایسه فقط روی یک دیتاست (اسنپ‌فود) انجام شد؛ دیتاست دوم (مرحله‌ی ۱۳، اختیاری) اجرا نشد، پس مقایسه‌ی بین‌دیتاستی در این نسخه از گزارش موجود نیست.

**۳. مزایا/معایب مدل تک‌زبانه در برابر چندزبانه**

ParsBERT: پارامترهای کمتر و اختصاصی برای فارسی، اما در این اجرا زودتر دچار overfitting شد. XLM-RoBERTa: در طول آموزش پایدارتر بود (افت accuracy بین epoch ۲ و ۳ فقط 0.8782 → 0.8776)، ولی مدل بزرگ‌تر و کندتری است و دانشش بین چند زبان تقسیم شده.

**۴. اثر تغییر learning rate**

با توجه به overfitting سریع ParsBERT حتی در همین 2e-5، احتمال می‌دهیم learning rate بزرگ‌تر (مثلاً 1e-3) بی‌ثباتی و افت شدیدتر ایجاد کند (Catastrophic Forgetting). learning rate کوچیک‌تر (مثلاً 1e-6) در ۳ epoch محدود، وقت کافی برای یادگیری نمی‌دهد و accuracy پایین‌تر می‌ماند. این فرضیه با اجرای واقعی آزمایش نشده است.


## جمع‌بندی و چک‌لیست تحویل

- [x] لینک نوت‌بوک Colab / فایل `.ipynb`
- [x] جدول مقایسه (بخش ۱.۵)
- [x] پاسخ سوالات تحلیلی (بخش ۱.۶)
- [x] لینک دیتاست: https://huggingface.co/datasets/PNLPhub/snappfood-sentiment-analysis
- [x] جمع‌بندی:

**جمع‌بندی نهایی:** در این تمرین یاد گرفتم که مدل تک‌زبانه بودن به‌تنهایی تضمین دقت بالاتر نیست — رفتار مدل در طول epochها (مثلاً overfitting) می‌تواند اثر بیشتری از زبان تخصصی مدل داشته باشد. همچنین با مشکل واقعی برچسب رشته‌ای (`label` به‌جای `label_id` عددی) مواجه شدم که باعث خطای مدل می‌شد و یاد گرفتم چطور با `cast_column` و `rename_column` آن را قبل از tokenization اصلاح کنم. اهمیت eval_strategy="epoch" هم برای من روشن‌تر شد: بدون آن، متوجه افت عملکرد ParsBERT در epoch سوم نمی‌شدم.

موفق باشی! 🌟
